In [1]:
import kwant
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.linalg import eigsh
import scipy.sparse as sp


In [2]:
#参数
dela = 1
t = 38 * dela
af = 400/0.1875* dela * 10**(-10)
a = 10**(-10)*(400/3)
mu = 8 * dela
U = 1.9 * dela
mz = 7 * dela
h = 1.2 * np.sqrt(mu**2 + dela**2)
L = 11
chaodaojiao = np.pi/10
saimanjiao = np.pi / 4

#矩阵信息
sx = np.array([[0, 1], [1, 0]], complex)
sy = np.array([[0, -1j], [1j, 0]], complex)
sz = np.array([[1, 0], [0, -1]], complex)
s0 = np.eye(2, dtype=complex)

#左边矩阵信息
HL_block=-(mu-2*t)*s0 + h*np.cos(0)*sx + h*np.sin(0)*sy
Delta_L=dela * np.exp(1j*chaodaojiao/2) * 1j * sy
H_L_onsite=np.block([
        [ HL_block,        Delta_L        ],
        [ Delta_L.conj().T, -HL_block.conj() ]
    ])
H_L_right_to_left_hop_block=-1*(t*s0+1j*af*sz/a)
H_L_right_to_left_hop=np.block([
        [ H_L_right_to_left_hop_block,        np.zeros((2,2))],
        [ np.zeros((2,2)), -H_L_right_to_left_hop_block.conj() ]
    ])

#中间矩阵信息
H_center_to_L=np.block([
        [ -t*s0,        np.zeros((2,2)) ],
        [ np.zeros((2,2)) , t*s0 ]
    ])
H_center_block=(U-mu+2*t)*s0+mz*sz
H_center=np.block([
        [ H_center_block,        np.zeros((2,2)) ],
        [ np.zeros((2,2)) , -H_center_block.conj()  ]
    ])
    
H_center_right_to_left_hop=np.block([
        [ -t*s0,        np.zeros((2,2)) ],
        [ np.zeros((2,2)) , t*s0 ]
    ])
H_R_to_center=np.block([
        [ -t*s0,        np.zeros((2,2)) ],
        [ np.zeros((2,2)) , t*s0 ]
    ])

#右边矩阵信息
HR_block=-(mu-2*t)*s0 + h*np.cos(saimanjiao)*sx + h*np.sin(saimanjiao)*sy
Delta_R=dela * np.exp(-1j*chaodaojiao/2) * 1j * sy
H_R_onsite=np.block([
        [ HR_block,        Delta_R        ],
        [ Delta_R.conj().T, -HR_block.conj() ]
    ])
H_R_right_to_left_hop_block=-1*(t*s0+1j*af/a*sz)
H_R_right_to_left_hop=np.block([
        [ H_R_right_to_left_hop_block,        np.zeros((2,2))],
        [ np.zeros((2,2)), -H_R_right_to_left_hop_block.conj() ]
    ])


In [10]:
import kwant
import numpy as np

# =========================
# 参数
# =========================
dela = 1
t = 38 * dela
af = 400/0.1875 * dela * 1e-10
a = 1e-10 * (400/3)
mu = 8 * dela
h = 1.2 * np.sqrt(mu**2 + dela**2)
chaodaojiao = np.pi / 10

sx = np.array([[0, 1], [1, 0]], complex)
sy = np.array([[0, -1j], [1j, 0]], complex)
sz = np.array([[1, 0], [0, -1]], complex)
s0 = np.eye(2, dtype=complex)

# =========================
# 左 lead Hamiltonian
# =========================
HL_block = -(mu - 2*t)*s0 + h*sx
Delta_L = dela * np.exp(1j*chaodaojiao/2) * 1j * sy

H_L_onsite = np.block([
    [ HL_block,           Delta_L ],
    [ Delta_L.conj().T,  -HL_block.conj() ]
])

H_L_hop_block = -(t*s0 + 1j*af/a*sz)
H_L_hop = np.block([
    [ H_L_hop_block, np.zeros((2,2)) ],
    [ np.zeros((2,2)), -H_L_hop_block.conj() ]
])

# =========================
# 中心 ↔ lead 耦合
# =========================
H_center_to_L = np.block([
    [ -t*s0, np.zeros((2,2)) ],
    [ np.zeros((2,2)),  t*s0 ]
])

# =========================
# Kwant lattice
# =========================
lat = kwant.lattice.chain(norbs=4)

# =========================
# 中心系统
# =========================
sys = kwant.Builder()
sys[lat(0)] = np.zeros((4,4), dtype=complex)

# =========================
# 左 lead（关键修正点）
# =========================
sym = kwant.TranslationalSymmetry([-1])
lead = kwant.Builder(sym)

lead[lat(0)] = H_L_onsite
lead[lat(0), lat(1)] = H_L_hop

# ⭐ 这里才是“中心 ↔ lead”的正确位置
lead[lat(0), lat(-1)] = H_center_to_L.conj().T

# attach
sys.attach_lead(lead)

sys = sys.finalized()

# =========================
# 计算自能
# =========================
E = 0.1 + 1e-6j
Sigma_L = sys.leads[0].selfenergy(E)

print("Sigma_L shape =", Sigma_L.shape)
print("Sigma_L =\n", Sigma_L)


Sigma_L shape = (4, 4)
Sigma_L =
 [[-30.12829583-1.22614344e+01j  -0.99418791+1.21946978e+01j
   -0.18933073+1.25013071e+00j  -0.08738684-2.40427902e-02j]
 [ -0.99418791+1.21946978e+01j -30.12829583-1.22614344e+01j
    0.08738684+2.40427902e-02j   0.18933073-1.25013071e+00j]
 [  0.20624741+1.24745137e+00j   0.09053945-4.13796610e-03j
   29.98060613-1.22011303e+01j   0.74694431+1.21368001e+01j]
 [ -0.09053945+4.13796610e-03j  -0.20624741-1.24745137e+00j
    0.74694431+1.21368001e+01j  29.98060613-1.22011303e+01j]]
